In [1]:
import cdms
import numpy as np
import os, sys 
import ROOT
from cats.cdataframe import CDataFrame
from CDMSDataCatalog import CDMSDataCatalog
import supercuts
import glob
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
%matplotlib inline

#CDMS = '/project/6049244/perry/CDMS/' # set in .bash_profile
#stylesheet = os.path.join(CDMS,"scripts","stylesheets","blues.mplstyle")
#plt.style.use(stylesheet)

#sys.path.append(os.path.join(os.path.join(CDMS,"scripts")))
#import setup

Welcome to JupyROOT 6.28/10


In [2]:
from scipy.optimize import curve_fit
from uncertainties import *
from uncertainties.umath import exp
import math

def func(x, a, b):
    return a*x + b

def fit_logPulse(x, y, start, stop):
    popt, pcov = curve_fit(func, x[start:stop], y[start:stop])
    return popt[0], np.sqrt(np.diag(pcov))[0], popt[1], np.sqrt(np.diag(pcov))[1]

def str_with_err(value, error):
    digits = -int(math.floor(math.log10(error)))
    return "{0:.{2}f}({1:.0f})".format(value, error*10**digits, digits)

### Propagate error for Geant4 SourceSim result

Found that simulating 1E9 Cf-252 decays in the CUTE fridge results in 9208 Ge-70 + n $\to$ Ge-71 interactions.  
Conservative uncertainty for Geant4 hadronic physics is 4%.  
Poisson uncertainty for total count.

In [3]:
def tot_err(count):
    poisson_err = np.sqrt(count)
    geant4_err = count * 0.04
    pos_err = count * 0.076

    err = np.sqrt(poisson_err ** 2 + geant4_err ** 2 + pos_err ** 2)
    decays = ufloat(count, err)
    
    return decays

In [4]:
print('Number of K shell decays', end=": ")
K_decays = tot_err(int(8059))
print('{:+.3uS}'.format(K_decays))

print('Number of L shell decays', end=": ")
L_decays = tot_err(int(959))
print('{:+.3uS}'.format(L_decays))

print('Number of M shell decays', end=": ")
M_decays = tot_err(int(177))
print('{:+.3uS}'.format(M_decays))

print('Total number of decays', end=": ")
total_decays = tot_err(int(9195))
print('{:+.3uS}'.format(total_decays))

Number of K shell decays: +8059(698)
Number of L shell decays: +959.0(88.0)
Number of M shell decays: +177.0(20.2)
Total number of decays: +9195(796)


In [5]:
K_over_tot = K_decays / total_decays * 100
print('{:+.1uS}'.format(K_over_tot))

L_over_tot = L_decays / total_decays * 100
print('{:+.1uS}'.format(L_over_tot))

M_over_tot = M_decays / total_decays * 100
print('{:+.1uS}'.format(M_over_tot))

+9(1)e+01
+10(1)
+1.9(3)


### Propagate errors in estimation of number of decays

[Cf-252 half-life](https://www.chemlin.org/isotope/californium-252)  
[Ge-71 half-life](https://www.chemlin.org/isotope/germanium-71)

In [6]:
# Z1 low bg 50V
series_list=['23240109_075338', '23240109_021236', 
'23240108_203134', '23231221_101235', 
'23231221_015705', '23231220_190923', 
'23231220_122140', '23231220_053358', 
'23231220_012745', '23231219_184002', 
'23231219_110331', '23231219_034952', 
'23231218_223530', '23231218_190035', 
'23231218_152721', '23231218_093255', 
'23231218_024511', '23231217_212512', 
'23231217_171613', 
'23231217_135018', '23231216_233807', 
'23231216_211119', '23231216_194929', 
'23231216_182937', '23231216_173436', 
'23231216_145300', '23231216_100125', 
'23231216_043946', '23231216_013604'] # Ge calibration

In [7]:
ProdTag = 'CUTE_T3GeCalib_NxM_P4.0.0_V05-06_C0.3.6'

In [8]:
filepath = [f'/scratch/perry/CDMS/CUTE/R37/Processed/Releases/{ProdTag}/Submerged/{ProdTag}_{i}.root' for i in series_list]

In [9]:
det = 1 # detector number

df = CDataFrame("rqDir/zip"+str(det), filepath, friends = [[x+":rqDir/eventTree" for x in filepath]])

In [10]:
## Apply some basic data quality filters and get the RQs you're interested in
logfile = '"cute_tower3testing.log"'
df = df.Define("LEDLogFile", logfile) 
df = df.CDefine("LEDOn", supercuts.ledOn_old, ["EventTime", "LEDLogFile"])
df = df.Filter("!LEDOn")
df_filtered = df.Filters(["TriggerType == 1", "TriggerDetectorNum=="+str(det), "PTOFamps>0"])

In [11]:
RQs = (["SeriesNumber", "PTOFamps", "EventNumber", "EventTime"])
df_rqs = df_filtered.AsNumpy(RQs)

In [12]:
import datetime
from zoneinfo import ZoneInfo

In [13]:
y2s = 3.156e7
d2s = 86400
m2s = 60

# Cf252 radioactive properties
Cf252_halflife = ufloat(2.645 * y2s, 0.008 * y2s) # s
Cf252_lifetime = Cf252_halflife / np.log(2) # s
Cf252_lambda   = 1 / Cf252_lifetime # Hz

# Ge-71 radioactive properties
Ge71_halflife = ufloat(11.468 * d2s, 0.008 * d2s) # s
Ge71_lifetime = Ge71_halflife / np.log(2) # s
Ge71_lambda   = 1 / Ge71_lifetime

In [14]:
Cf252_activity = ufloat(37000, 37000 * 0.15)                          # activity of Cf252 at reference time; given in Hz
t_ref          = datetime.datetime(2020, 3, 15, 00, 0, tzinfo=ZoneInfo("America/New_York")).timestamp()    # reference time of nominal activity; Unix time
activ_prob     = total_decays * 1e-9                                  # Ge activations per neutron emitted; found using sourcesim w/ CUTE geometry

# rate of Ge activation during exposure
A = Cf252_activity * activ_prob # activations / second

In [16]:
start_times = pd.read_csv('../source_exposure_start_times.csv')
end_times = pd.read_csv('../source_exposure_end_times.csv')

timestamp_err = 30 * m2s

exposures = {i: {} for i in range(8)}

for i in range(8):
    year_0, year_f   = start_times['year'][i], end_times['year'][i]
    month_0, month_f = start_times['month'][i], end_times['month'][i]
    day_0, day_f     = start_times['day'][i], end_times['day'][i]
    hour_0, hour_f   = start_times['hour'][i], end_times['hour'][i]
    min_0, min_f     = start_times['minute'][i], end_times['minute'][i]
    s_0, s_f         = start_times['second'][i], end_times['second'][i]
    exposures[i]['t0'] = ufloat(datetime.datetime(year_0, month_0, day_0, hour_0, min_0, s_0, tzinfo=ZoneInfo("America/New_York")).timestamp(), timestamp_err)
    exposures[i]['tf'] = ufloat(datetime.datetime(year_f, month_f, day_f, hour_f, min_f, s_f, tzinfo=ZoneInfo("America/New_York")).timestamp(), timestamp_err)
    exposures[i]['dt'] = exposures[i]['tf'] - exposures[i]['t0']

In [17]:
# find total decays with N = int[ A exp( -lambda_Cf252 (t - tref) ) ] dt
# t from t0 to T
def integrate_decays(T, t0):
    return (Cf252_activity * (-exp(Cf252_lambda * (t_ref - T)) + exp(Cf252_lambda * (t_ref - t0)))) / Cf252_lambda

In [18]:
#### starting with 0 decays ####
N_tot = 0

#### track change in number of Cf-252 decays during exposures ####
for i in range(len(exposures)):
    N = integrate_decays(exposures[i]['tf'], exposures[i]['t0'])

    N_tot += N

print('{:+.3uS} total Cf-252 decays'.format(N_tot))

+1.637(246)e+10 total Cf-252 decays


In [19]:
# find total activations with N = int[ A exp( -lambda_Cf252 (t + t0) ) * exp(-lambda_Ge71 (T - t) ) ] dt + N0 exp(-lambda_Ge71 T) 
# t from t0 to T
def integrate_activations(N, T, t0):
    return (A / (Cf252_lambda - Ge71_lambda) * ( exp(- Cf252_lambda * t0 - Ge71_lambda * T) - exp(-Cf252_lambda * (T + t0)) ) + 
            N * exp(-Ge71_lambda * T) )

In [20]:
#### starting with 0 nuclei activated ####
N = 0

#### track change in number of Ge-71 nuclei during and after exposures ####
for i in range(len(exposures)):
    N = integrate_activations(N, exposures[i]['dt'], exposures[i]['t0'] - t_ref)
    if (i != len(exposures) - 1):
        N = N * exp(-Ge71_lambda * (exposures[i+1]['t0'] - exposures[i]['tf']))
    else:
        # check change in number of Ge-71 during each series
        N_tot = 0
        for sn in np.unique(df_rqs['SeriesNumber']):
            snCut = df_rqs['SeriesNumber'] == sn
            evtTimes = df_rqs['EventTime'][snCut]
            sn_start = min(evtTimes)
            sn_end = max(evtTimes)

            N_start = N * exp(-Ge71_lambda * (sn_start - (exposures[len(exposures) - 1]['t0'] + exposures[len(exposures) - 1]['dt'])))
            N_end = N * exp(-Ge71_lambda * (sn_end - (exposures[len(exposures) - 1]['t0'] + exposures[len(exposures) - 1]['dt'])))
            #print(f'{np.round(N_start - N_end)} Ge-71 events in series {int(sn)}')

            N_tot += N_start - N_end
            print(f'{sn}: {N_tot}')

print()
print('{:+.3uS} total Ge-71 events'.format(N_tot))
print('{:+.3uS} K shell events'.format(N_tot * K_over_tot / 100))
print('{:+.3uS} L shell events'.format(N_tot * L_over_tot / 100))
print('{:+.3uS} M shell events'.format(N_tot * M_over_tot / 100))

23231216013604.0: 153+/-27
23231216043946.0: (4.3+/-0.7)e+02
23231216100125.0: (4.6+/-0.8)e+02
23231216145300.0: (6.1+/-1.1)e+02
23231216173436.0: (6.5+/-1.1)e+02
23231216182937.0: (7.1+/-1.2)e+02
23231216194929.0: (7.2+/-1.3)e+02
23231216211119.0: (7.8+/-1.4)e+02
23231216233807.0: (8.0+/-1.4)e+02
23231217135018.0: (9.4+/-1.6)e+02
23231217171613.0: (1.12+/-0.20)e+03
23231217212512.0: (1.37+/-0.24)e+03
23231218024511.0: (1.68+/-0.29)e+03
23231218093255.0: (1.88+/-0.33)e+03
23231218152721.0: (2.03+/-0.35)e+03
23231218190035.0: (2.1+/-0.4)e+03
23231218223530.0: (2.3+/-0.4)e+03
23231219034952.0: (2.7+/-0.5)e+03
23231219110331.0: (3.0+/-0.5)e+03
23231219184002.0: (3.2+/-0.6)e+03
23231220012745.0: (3.4+/-0.6)e+03
23231220053358.0: (3.7+/-0.6)e+03
23231220122140.0: (4.0+/-0.7)e+03
23231220190923.0: (4.2+/-0.7)e+03
23231221015705.0: (4.5+/-0.8)e+03
23231221101235.0: (4.5+/-0.8)e+03
23240108203134.0: (4.6+/-0.8)e+03
23240109021236.0: (4.7+/-0.8)e+03
23240109075338.0: (4.7+/-0.8)e+03

+4669(813)

In [21]:
ufloat(177, 177 * 0.04) + ufloat(959, 959 * 0.04) + ufloat(8059, 8059 * 0.04)

9195.0+/-324.71154214163687

In [22]:
ufloat(4432, 67) / ufloat(3727, np.sqrt(649**2 - (3727 * 0.15)**2))

1.1891601824523745+/-0.10670302478932454